# Architecture Comparison

4 architectures: Qwen2.5-1.5B, Llama-3.2-1B, Gemma-2-2B, SmolLM2-1.7B

In [ ]:
import sys; sys.path.insert(0, '../..')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collection.track1_ggml.profiler_wrapper import parse_profiler_output

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300})

In [ ]:
configs = [
    ('orin_cpu','qwen2.5-1.5b','qwen','../../data/raw/orin_cpu/orin_cpu_qwen2.5_1.5b_profile.jsonl'),
    ('orin_cpu','llama-3.2-1b','llama','../../data/raw/orin_cpu_llama/orin_cpu_llama_1b_profile.jsonl'),
    ('orin_cpu','gemma-2-2b','gemma','../../data/raw/orin_cpu_gemma/orin_cpu_gemma_2b_profile.jsonl'),
    ('orin_cpu','smollm2-1.7b','smollm','../../data/raw/orin_cpu_smollm/orin_cpu_smollm_1.7b_profile.jsonl'),
    ('op11_cpu','qwen2.5-1.5b','qwen','../../data/raw/op11_cpu/op11_cpu_profile.jsonl'),
    ('op11_cpu','llama-3.2-1b','llama','../../data/raw/op11_cpu_llama/op11_llama_1b_profile.jsonl'),
    ('op11_cpu','gemma-2-2b','gemma','../../data/raw/op11_cpu_gemma/op11_gemma_2b_profile.jsonl'),
    ('op11_cpu','smollm2-1.7b','smollm','../../data/raw/op11_cpu_smollm/op11_smollm_1.7b_profile.jsonl'),
]

all_records = []
for device, model, arch, path in configs:
    records = parse_profiler_output(path, f'{device}_{arch}', device, model, arch, 'q4_k_m', 6)
    all_records.extend(records)
df = pd.DataFrame([r.model_dump() for r in all_records])
print(f'Total: {len(df):,} records')

## Sublayer Distribution by Architecture

In [ ]:
MAIN = ['attention_qkv','flash_attention','attention_out','ffn_gate','ffn_up','ffn_down','lm_head','rmsnorm']
df_main = df[df['sublayer'].isin(MAIN) & (df['phase']=='decode')]

for device in ['orin_cpu','op11_cpu']:
    dev = df_main[df_main['device']==device]
    pivot = dev.groupby(['architecture','sublayer'])['latency_us'].sum().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(12,5))
    pivot_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    ax.set_ylabel('Latency %'); ax.set_title(f'{device} — Architecture Sublayer Distribution')
    ax.legend(bbox_to_anchor=(1.02,1), fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../../claudedocs/figures/nb2_arch_{device}.pdf', bbox_inches='tight')
    plt.show()